# ETL del archivo en crudo `cast.parquet`

## Librerías

In [258]:
import pandas as pd
import ast
import os
import gc

## Extracción

In [259]:
url = "https://github.com/FranciscoHugoLezik/Movies_data/blob/main/credits/cast.parquet?raw=true"

raw_cast = (
    pd.read_parquet(
        url, 
        engine='fastparquet'
        )
    )

Primeras cinco filas

In [260]:
raw_cast.head()

,cast,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...",11862


Ver, mas en detalle, un dato de la columna 'cast'.

In [261]:
raw_cast.iloc[0]['cast']

"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}, {'cast_id': 17, 'character': 'Slinky Dog (voice)', 'credit_id': '52fe4284c3a36847f8024fa1', 'gender': 2, 'id': 12899, 'name': 'Jim Varney', 'order': 3, 'profile_path': '/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg'}, {'cast_id': 18, 'character': 'Rex (voice)', 'credit_id': '52fe4284c3a36847f8024fa5', 'gender': 2, 'id': 12900, 'name': 'Wallace Shawn', 'order': 4, 'profile_path': '/oGE6JqPP2xH4t

Tamaño

In [262]:
raw_cast.shape

(45476, 2)

Los datos están completos.

In [263]:
raw_cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cast    45476 non-null  object
 1   id      45476 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 710.7+ KB


## Transformación

### Borrar duplicados en la columna 'id'

Hay duplicados

In [264]:
raw_cast['id'].duplicated().any()

True

Se eliminan los duplicados

In [265]:
nested_cast = (
    raw_cast
    .drop_duplicates(
        subset='id'
        )
    )

In [266]:
nested_cast['id'].duplicated().any()

False

In [267]:
del raw_cast
gc.collect()

102

### Renombrar el nombre de la columna 'id' por 'movie_id'

Se hace esto porque dentro de los valores anidados hay una clave llamada 'id' y para, mas adelante, poder hacer un join con la tabla de las peliculas (movies.parquet).

In [268]:
nested_cast.rename(
    columns={'id': 'movie_id'}, 
    inplace=True
    )

In [269]:
nested_cast.columns

Index(['cast', 'movie_id'], dtype='object')

In [270]:
nested_cast.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 45432 entries, 0 to 45475
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   cast      45432 non-null  object
 1   movie_id  45432 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.0+ MB


No hay valores nulos

In [271]:
nested_cast.isnull().sum()

cast        0
movie_id    0
dtype: int64

### Desanidar la columna 'cast' y unirla a la columna 'movie_id'

In [272]:
elenco_por_pelicula = [
    {
        **actores, 
        'movie_id': 
            row['movie_id']
            } 
    for _, row 
    in nested_cast.iterrows() 
    for actores 
    in ast.literal_eval(
        row['cast']
        )
    ]

In [273]:
denested_cast = pd.DataFrame(elenco_por_pelicula)

In [274]:
del nested_cast
del elenco_por_pelicula
gc.collect()

286

Primeras cinco filas.

In [275]:
denested_cast.head()

,cast_id,character,credit_id,gender,id,name,order,profile_path,movie_id
0,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg,862
1,15,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,2,12898,Tim Allen,1,/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg,862
2,16,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2,7167,Don Rickles,2,/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg,862
3,17,Slinky Dog (voice),52fe4284c3a36847f8024fa1,2,12899,Jim Varney,3,/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg,862
4,18,Rex (voice),52fe4284c3a36847f8024fa5,2,12900,Wallace Shawn,4,/oGE6JqPP2xH4tNORKNqxbNPYi7u.jpg,862


Primera fila.

In [276]:
denested_cast.iloc[0]

cast_id                                       14
character                          Woody (voice)
credit_id               52fe4284c3a36847f8024f95
gender                                         2
id                                            31
name                                   Tom Hanks
order                                          0
profile_path    /pQFoyx7rp09CJTAb932F2g8Nlho.jpg
movie_id                                     862
Name: 0, dtype: object

In [277]:
denested_cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 562044 entries, 0 to 562043
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   cast_id       562044 non-null  int64 
 1   character     562044 non-null  object
 2   credit_id     562044 non-null  object
 3   gender        562044 non-null  int64 
 4   id            562044 non-null  int64 
 5   name          562044 non-null  object
 6   order         562044 non-null  int64 
 7   profile_path  388366 non-null  object
 8   movie_id      562044 non-null  int64 
dtypes: int64(5), object(4)
memory usage: 38.6+ MB


In [278]:
denested_cast.isnull().sum()

cast_id              0
character            0
credit_id            0
gender               0
id                   0
name                 0
order                0
profile_path    173678
movie_id             0
dtype: int64

### Renombrar el nombre de la columna 'id' por 'person_id'

Se hace esto para que se identifique rapidamente que es el id de la persona.

In [279]:
denested_cast.rename(
    columns={'id': 'person_id'}, 
    inplace=True
    )

In [280]:
('id' not in denested_cast.columns 
 and 'person_id' in denested_cast.columns)

True

### Eliminar la columnas 'credit_id', 'order' y 'profile_path'

Se hace esto porque son innecesarias en el contexto de la visualizacion de las peliculas por parte de los usuarios.

In [281]:
innecesarias = [
    'credit_id', 
    'order', 
    'profile_path'
]

In [282]:
denested_cast.drop(
    columns=innecesarias, 
    inplace=True
    )

KeyError: "[('credit_id', 'order', 'profile_path')] not found in axis"

In [257]:
estan_eliminadas = True
for i in innecesarias:
    if i in denested_cast.columns:
        estan_eliminadas = False
        break
estan_eliminadas

SyntaxError: incomplete input (3017113826.py, line 3)

### Se cambia el tipo de las columnas id y de gender a str

Se hace esto porque los id son etiquetas y gender es un dato categorico. El resto de los datos tienen el tipo correcto que es str.

In [249]:
denested_cast.dtypes

cast_id       int64
character    object
credit_id    object
gender        int64
person_id     int64
name         object
order         int64
movie_id      int64
dtype: object

In [250]:
denested_cast['cast_id'] = (
    denested_cast['cast_id']
    .astype(str)
    )
denested_cast['gender'] = (
    denested_cast['gender']
    .astype(str)
    )
denested_cast['movie_id'] = (
    denested_cast['movie_id']
    .astype(str)
    )
denested_cast['person_id'] = (
    denested_cast['person_id']
    .astype(str)
    )

In [251]:
denested_cast.dtypes

cast_id      object
character    object
credit_id    object
gender       object
person_id    object
name         object
order        object
movie_id     object
dtype: object

## Carga

In [252]:
directorio_actual = (
    os.getcwd()
    )
directorio_actual

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\notebooks\\ETL'

In [253]:
directorio_del_proyecto = (
    os.path.dirname(
        os.path.dirname(
            directorio_actual
            )
        )
    )
directorio_del_proyecto

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas'

In [254]:
directorio_a_exportar = (
    os.path.join(
        directorio_del_proyecto, 
        'data', 
        'ETL', 
        'cast.parquet'
        )
    )
directorio_a_exportar

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\data\\ETL\\cast.parquet'

In [255]:
denested_cast.to_parquet(
    directorio_a_exportar
    )

In [256]:
del denested_cast
gc.collect()

120